<a href="https://colab.research.google.com/github/Anastasiia-hub0/unsolved-homicides-analysis/blob/main/unsolved_homicides_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsolved Homicides: Which Case Characteristics Are Associated with a Case Staying Unsolved?
**Module:** Data Analytics (DBMS71008)- CA2  
**Student:** Anastasiia Tkachenko  
**Dataset:** Murder Accountability Project (MAP) – case-level Supplementary Homicide Report  
**GitHub:** https://github.com/Anastasiia-hub0/unsolved-homicides-analysis

# 1. Business Understanding

## 1.1 Context and problem
In the USA, many murder cases are never solved. Police do not have unlimited investigators and time, so they have to choose which cases need more attention.

In this project I want to find out which things we know about a case at the start (for example place, year, weapon, age and sex of the victim) are connected with the case staying unsolved.

**Research question:** Which characteristics of a homicide case that are known when it is reported are associated with the case staying unsolved, and can we use them to predict if the case will be solved?

**Unit of analysis:** one row = one murder case.

## 1.2 Stakeholders and users
- **Who gives the data:** the FBI reports, collected by the Murder Accountability Project (MAP).
- **Who uses the result:** the police and the people who decide how to share resources between murder cases. They decide which cases need more investigators and more time.
- **Who is affected:** the families and friends of the victims. They will not use the model, but the result matters to them. They want the offender to be found and punished by law. Also people in areas where cases are solved less often.

## 1.3 Business objective
Help to decide which murder cases need extra attention, so fewer cases stay unsolved.

## 1.4 Analytics objective
This is a binary classification task. The model predicts if a case is solved or unsolved, using only information known when the case is reported.

- **Target:** solved or unsolved.
- **Positive class = unsolved = 1.** Positive does not mean something good. It means the thing we are looking for. We look for unsolved cases, because these are the cases that need attention. Solved cases are already solved, so they are class 0.

## 1.5 Success criteria
**Technical**
- The model should be better than the baseline (the baseline always says "solved") in recall and F1 for unsolved cases. The baseline gets zero in both.
- The main metric is recall for the unsolved class. I also check precision and F1, so the list of risky cases is not too big and useless.
- I do not use accuracy alone. If most cases are solved, a model that always says "solved" has high accuracy but finds no unsolved cases.

**Practical**
- The list of risky cases should be easy to explain, so we can say which factors make a case more likely to stay unsolved.
- The model only helps. A human makes the final decision.

## 1.6 Which mistake is worse
- **Worse mistake:** the model says the case will be solved, but it stays unsolved. This case will not get extra resources, and the victim's family gets no justice.
- **Smaller mistake:** the model says the case will stay unsolved, but it would have been solved. We just spend a bit of extra attention.

So missing an unsolved case is worse than a false alarm. That is why recall is the most important metric here.

## 1.7 Constraints and assumptions
- **Time:** the project is due on 4 December 2026.
- **Old data:** the data is historical, so it may not show how police work now.
- **Not complete:** not every murder is reported to the FBI.
- **Data leakage:** columns about the offender (age, sex, relationship) are probably filled only for solved cases. We do not know them when a case starts, so I should not use them as inputs. I will check this in Data Understanding.
- **Missing values** can be written as "Unknown" or 999, not as empty cells.
- **Class imbalance** is possible. I will check it.
- **Ethics:** the data has victim race and location, so I need to talk about bias and fairness. The results show a connection, not a cause.
- **Tools:** Google Colab, Python and GitHub.
- **Licence:** I need to write the source and licence of the data.

## 1.8 Initial project plan
I will follow the six CRISP-DM phases:
1. **Business Understanding** – this section.
2. **Data Understanding** – source, what one row means, variables, data quality (missing values, "Unknown", leakage), balance of solved and unsolved.
3. **Data Preparation** – choose columns, clean, encode. I will keep a preparation log.
4. **Modelling** – a simple baseline first, then a decision tree. I may also compare it with logistic regression. Train/test split with a fixed random seed.
5. **Evaluation** – confusion matrix, precision, recall, F1, mistakes of the model, bias and ethics.
6. **Deployment** – how the police could use a list of risky cases, what should not be automated, future work.

# 2. Data Understanding

## 2.1 Initial Data Collection Report

**Data source:** The data comes from the Murder Accountability Project (MAP), a non-profit organisation that collects homicide data in the USA. I downloaded the file from their website (murderdata.org, page "Data & Docs"). The file is called `SHR76_25a.csv`.

**Date of access:** I downloaded the file on 17 October 2026.

**Licence:** I could not find any licence or terms of use on the MAP website. The page only says "Copyright 2019 by Murder Accountability Project", and that the files are available to police and the general public. I will name MAP as the source in my references. I will not upload the dataset to GitHub, I will only explain how to download it.

**How the data was collected:** The main part is the FBI Supplementary Homicide Report (SHR), which police agencies in the USA send to the FBI. MAP also added nearly 39,000 homicides that it got through the Freedom of Information Act, because they were not reported to the FBI.

**Population, time and place:** Homicide cases in the USA, from 1976 to 2025.

**Size and files:** One file with 929,433 rows and 31 columns. There is only one table, so nothing needs to be joined.

**Known limits:** On the MAP website it is written that nearly 39,000 homicides were not reported to the Justice Department, and MAP got them with the Freedom of Information Act. This means the FBI reports alone are not complete, so some cases can be missing even in this file. I also have to check how many values are "Unknown".

In [1]:
import pandas as pd

df = pd.read_csv("SHR76_25a.csv", low_memory=False)
df.head()

,ID,CNTYFIPS,Ori,State,Agency,Agentype,Source,Solved,Year,Month,...,OffEthnic,Weapon,Relationship,Circumstance,Subcircum,VicCount,OffCount,FileDate,MSA,filter_$
0,197609001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,No,1976,September,...,Unknown or not reported,Other or type unknown,Relationship not determined,Other,NaN,0,0,30180.0,"Montgomery, AL",Not Selected
1,197701001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,Yes,1977,January,...,Unknown or not reported,Other or type unknown,Acquaintance,Brawl due to influence of alcohol,NaN,0,0,30180.0,"Montgomery, AL",Not Selected
2,197703001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,Yes,1977,March,...,Unknown or not reported,"Handgun - pistol, revolver, etc",Acquaintance,Other arguments,NaN,0,1,30180.0,"Montgomery, AL",Not Selected
3,197703001AL00401,"Autauga, AL",AL00401,Alabama,Prattville,Municipal police,FBI,Yes,1977,March,...,Unknown or not reported,Shotgun,Husband,Other arguments,NaN,0,0,30180.0,"Montgomery, AL",Not Selected
4,197708001AL00400,"Autauga, AL",AL00400,Alabama,Autauga County,Sheriff,FBI,Yes,1977,August,...,Unknown or not reported,Knife or cutting instrument,Acquaintance,Circumstances undetermined,NaN,0,0,30180.0,"Montgomery, AL",Not Selected


In [2]:
df.shape

(929433, 31)

**What this shows:** The dataset has 929,433 rows and 31 columns. One row is one homicide case, so the unit of analysis is a case. This is a lot of data, enough for a train/test split and a model. The data covers the years 1976–2025.

In [3]:
df.columns

Index(['ID', 'CNTYFIPS', 'Ori', 'State', 'Agency', 'Agentype', 'Source',
       'Solved', 'Year', 'Month', 'Incident', 'ActionType', 'Homicide',
       'Situation', 'VicAge', 'VicSex', 'VicRace', 'VicEthnic', 'OffAge',
       'OffSex', 'OffRace', 'OffEthnic', 'Weapon', 'Relationship',
       'Circumstance', 'Subcircum', 'VicCount', 'OffCount', 'FileDate', 'MSA',
       'filter_$'],
      dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 929433 entries, 0 to 929432
Data columns (total 31 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   ID            929433 non-null  object 
 1   CNTYFIPS      929433 non-null  object 
 2   Ori           929433 non-null  object 
 3   State         929433 non-null  object 
 4   Agency        929433 non-null  object 
 5   Agentype      929433 non-null  object 
 6   Source        929433 non-null  object 
 7   Solved        929433 non-null  object 
 8   Year          929433 non-null  int64  
 9   Month         929433 non-null  object 
 10  Incident      929433 non-null  int64  
 11  ActionType    929433 non-null  object 
 12  Homicide      929433 non-null  object 
 13  Situation     929433 non-null  object 
 14  VicAge        929433 non-null  int64  
 15  VicSex        929433 non-null  object 
 16  VicRace       929433 non-null  object 
 17  VicEthnic     929433 non-null  object 
 18  OffA

**What this shows:** The table has 31 columns: 24 are text and 7 are numbers. The target is `Solved`.

Only two columns have really empty values: `Subcircum` (almost all empty) and `FileDate` (about 24,900 empty). Other columns look full, but unknown values may be written as "Unknown" or 999, so I need to check this.

Columns about the offender (`OffAge`, `OffSex`, `OffRace`, `Relationship` and others) are known only after the offender is found. They may cause data leakage, so I will check them before choosing features.

## 2.2 Data Description Report
Here I look at what is inside each column: example values, data types and the range of the numbers.

In [5]:
df.head().T

,0,1,2,3,4
ID,197609001AL00400,197701001AL00400,197703001AL00400,197703001AL00401,197708001AL00400
CNTYFIPS,"Autauga, AL","Autauga, AL","Autauga, AL","Autauga, AL","Autauga, AL"
Ori,AL00400,AL00400,AL00400,AL00401,AL00400
State,Alabama,Alabama,Alabama,Alabama,Alabama
Agency,Autauga County,Autauga County,Autauga County,Prattville,Autauga County
Agentype,Sheriff,Sheriff,Sheriff,Municipal police,Sheriff
Source,FBI,FBI,FBI,FBI,FBI
Solved,No,Yes,Yes,Yes,Yes
Year,1976,1977,1977,1977,1977
Month,September,January,March,March,August


**What this shows:** Most columns are text. `Solved` (Yes/No) is my target. `ID`, `Ori` and `Agency` are just codes and names, so I won't use them to predict.

In the first case (`Solved = No`) the offender is unknown: `OffAge = 999` and `OffSex = Unknown`. So 999 means "unknown", and offender columns may give away the answer (data leakage). Unknown values are written as text ("Unknown", "Other or type unknown"), not as empty cells.

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Year,929433.0,1999.826851,14.694502,1976.0,1987.0,1998.0,2013.0,2025.0
Incident,929433.0,51.409443,775.159555,0.0,1.0,2.0,11.0,27113.0
VicAge,929433.0,47.316518,116.987958,0.0,22.0,30.0,43.0,999.0
OffAge,929433.0,347.816624,454.100249,0.0,24.0,38.0,999.0,999.0
VicCount,929433.0,0.136712,0.605979,0.0,0.0,0.0,0.0,52.0
OffCount,929433.0,0.199429,0.628601,0.0,0.0,0.0,0.0,40.0
FileDate,904562.0,54666.859729,32811.728070,1121.0,30180.0,41692.0,82212.0,123197.0


**What this shows:** `Year` is from 1976 to 2025. `VicAge` and `OffAge` have max 999, which means "unknown", so their mean is wrong. For `OffAge` even the 75% value is 999, so at least a quarter of cases have unknown offender age. `Incident` and `FileDate` are technical numbers and not useful for my question.

### Data dictionary
Source: MAP Data Dictionary for the amplified Supplementary Homicide Report (`MAPdefinitionsSHR.pdf`).

| Column | Meaning | Values / notes | Known when case is reported? |
|---|---|---|---|
| ID | Case ID made from year, month, incident and agency | Victims of the same incident have the same ID | Yes (identifier, not used) |
| CNTYFIPS | County and state of the police agency | e.g. "Autauga, AL" | Yes |
| Ori | Code of the police agency | e.g. AL00400 | Yes (identifier, not used) |
| State | State of the police agency | e.g. Alabama | Yes |
| Agency | Name of the police agency | e.g. Autauga County | Yes (identifier, not used) |
| Agentype | Type of police agency | Sheriff, Municipal police, State police... | Yes |
| Source | Where the record came from | FBI or FOIA (got by MAP) | Yes |
| **Solved** | **Was the offender identified when the report was made** | **Yes / No - target** | - |
| Year | Year of the homicide (or when the body was found) | 1976–2025 | Yes |
| Month | Month of the homicide | January–December | Yes |
| Incident | Case number inside the month | Technical number | Yes (not used) |
| ActionType | Normal update or adjustment of an old report | | Yes (technical) |
| Homicide | Murder or manslaughter by negligence | | Yes |
| Situation | Single/multiple victims and offenders | Includes "unknown offender" | **No – leakage** |
| VicAge | Age of the victim | 0 = under 1 year, 99 = 99+, 999 = unknown | Yes |
| VicSex | Sex of the victim | Male, Female, Unknown | Yes |
| VicRace | Race of the victim | Asian, Black, American Indian, White, Unknown | Yes (sensitive) |
| VicEthnic | Hispanic or not | Many agencies do not report it | Yes (sensitive) |
| OffAge | Age of the offender | 999 = offender not identified | **No – leakage** |
| OffSex | Sex of the offender | Unknown = usually not identified | **No – leakage** |
| OffRace | Race of the offender | Unknown = usually not identified | **No – leakage** |
| OffEthnic | Hispanic or not (offender) | Often not reported | **No - leakage** |
| Weapon | Weapon used | Handgun, knife, ... "Other or type unknown" | Yes |
| Relationship | Relationship victim–offender | "Relationship not determined" if unknown | **No – leakage** |
| Circumstance | Circumstances of the crime | "Circumstances undetermined" if unknown | **No – likely leakage** |
| Subcircum | Extra info when the victim was a felon | Mostly empty | **No** |
| VicCount | Number of additional victims | 0 = only one victim | Yes |
| OffCount | Number of additional offenders | 0 = only one offender | **No – leakage** |
| FileDate | Date the record was reported (not crime date) | Sometimes estimated, some missing | Not useful |
| MSA | Metropolitan area | e.g. "Rural Alabama" | Yes |
| filter_$ | Not in the data dictionary | Probably a leftover from SPSS | Not used |

**What this shows:** The data dictionary confirms that "solved" means the offender was identified when the report was made. It also says that `OffAge` = 999 and "Unknown" offender sex or race are used when the offender was not identified. So all offender columns, and also `Situation` and `Relationship`, are only known after a case is solved. I will not use them as predictors, because this would be data leakage. Race and ethnicity columns are sensitive, so I need a clear reason if I use them.

## 2.3 Data Exploration Report

### Class balance
First I check how many cases are solved and how many are not. If one class is much bigger, a model that always guesses that class can have high accuracy but find no unsolved cases.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

class_counts = df["Solved"].value_counts()
class_percent = df["Solved"].value_counts(normalize=True).mul(100).round(1)

class_summary = pd.DataFrame({
    "count": class_counts,
    "percentage": class_percent,
})
display(class_summary)

ax = sns.countplot(data=df, x="Solved")
ax.set_title("Homicide case class distribution")
ax.set_xlabel("Solved")
ax.set_ylabel("Number of cases")
plt.show()

,count,percentage
Solved,,
Yes,657639,70.8
No,271794,29.2


**What this shows:** 70.8% of cases are solved and 29.2% are not. The classes are not balanced, so a model that always says "solved" gets 71% accuracy but finds no unsolved cases. That's why I use recall and F1, and I will keep the same balance in train and test.

### Type of homicide
I check how many cases are intentional and how many are by negligence.

In [ ]:
df["Homicide"].value_counts()

**What this shows:** 905,648 cases are intentional homicides and 23,785 (about 2.6%) are manslaughter by negligence, like hunting accidents. My question is about intentional killings, so I will remove the negligence cases in Data Preparation. Most of the data stays.

### Weapon
I check which weapons are used most often, and in which cases more homicides stay unsolved.

In [ ]:
df["Weapon"].value_counts()

**What this shows:** Handguns are the most common weapon (442,140 cases, almost half). All firearms together are about two thirds of the cases. Knives are second. Unknown weapons are written in two ways: "Other or type unknown" and "Weapon Not Reported". The second one is not in the data dictionary. Some groups are very small (for example "Pushed or thrown out window", 262 cases), so I may group them in Data Preparation.

In [ ]:
df["Unsolved"] = df["Solved"].map({"Yes": 0, "No": 1})

unsolved_by_weapon = (
    df.groupby("Weapon")
    .agg({"Unsolved": "mean"})
    .sort_values(by="Unsolved", ascending=False)
    .reset_index()
)

sns.barplot(x="Unsolved", y="Weapon", data=unsolved_by_weapon, color="b");

**What this shows:** On average 29% of cases are unsolved. Strangulation and guns with unknown type have more unsolved cases (about 45%), knives and beating have fewer (16–20%). So the weapon is linked to the result, but this is only an association, not a cause.

This is not always true, because a weapon can be found even if the offender is not, so it is only a tendency.

### Unsolved cases by year
I check if the share of unsolved cases changes over time.

In [ ]:
unsolved_by_year = (
    df.groupby("Year")
    .agg({"Unsolved": "mean"})
    .reset_index()
)

sns.lineplot(data=unsolved_by_year, x="Year", y="Unsolved", marker="o")
plt.title("Share of unsolved homicides by year")
plt.show()

**What this shows:** The share of unsolved cases grew from about 21% in 1976 to about 33% in the early 1990s, then stayed around 30–34%. After 2021 it suddenly drops to about 17% in 2025. This drop looks strange and may be a reporting problem (for example, a change in how police report to the FBI), not real improvement. I will check the number of cases per year in the Data Quality Report.

### Unsolved cases by victim sex
I check if the share of unsolved cases is different for male and female victims.

In [ ]:
unsolved_by_sex = (
    df.groupby("VicSex")
    .agg({"Unsolved": "mean"})
    .reset_index()
)

sns.barplot(x="VicSex", y="Unsolved", data=unsolved_by_sex, color="b");

**What this shows:** About 22% of cases with female victims are unsolved, and about 31% with male victims. When the victim's sex is unknown, more than half (56%) are unsolved. The data dictionary says "Unknown" is used when only incomplete remains were found, so these cases are probably very hard. This is an association, not a cause.

### Victim age
I compare the age of victims in solved and unsolved cases. I leave out age 999, because it means "unknown".

In [ ]:
ages_known = df.loc[df["VicAge"] < 999]

sns.histplot(data=ages_known, x="VicAge", hue="Solved", bins=20);

**What this shows:** Most victims are 15–35 years old, and in this group about a third of cases are unsolved. Cases with young children (0–5) are almost always solved (only about 10% unsolved). For older victims about a quarter of cases are unsolved. So victim age is linked to whether a case is solved.

### Unsolved cases by state
I check if the share of unsolved cases is different in different states.

In [ ]:
unsolved_by_state = (
    df.groupby("State")
    .agg({"Unsolved": "mean"})
    .sort_values(by="Unsolved", ascending=False)
    .reset_index()
)

plt.figure(figsize=(8, 12))
sns.barplot(x="Unsolved", y="State", data=unsolved_by_state, color="b");

**What this shows:** The share of unsolved cases is very different between states: from about 5% (North Dakota) to about 60% (District of Columbia). States with big cities, like New York, Maryland, Illinois and California, have more unsolved cases. Rural states, like the Dakotas, Montana and Idaho, have fewer. So location is strongly linked to whether a case is solved, but this does not show the reason.

## 2.4 Data Quality Report

### Missing values
First I check how many empty cells each column has.

In [ ]:
missing_values = (df.isnull().sum() / len(df)).sort_values(ascending=False)
plt.figure(figsize=(8, 10))
sns.barplot(x=missing_values.values, y=missing_values.index, color='b');

**What this shows:** Only two columns have empty cells: `Subcircum` (about 96% empty) and `FileDate` (about 3% empty). All other columns are full. `Subcircum` is almost empty and `FileDate` is not needed for my question, so I will not use them. But unknown values can also be hidden as text or 999, so I check this next.

### Hidden missing values
Some unknown values are not empty cells but are written as 999 or "Unknown". I count them in the columns I may use as features.

In [ ]:
print("VicAge = 999:", (df["VicAge"] == 999).sum())
print("VicSex = Unknown:", (df["VicSex"] == "Unknown").sum())
print("VicRace = Unknown:", (df["VicRace"] == "Unknown").sum())
print("VicEthnic = Unknown or not reported:", (df["VicEthnic"] == "Unknown or not reported").sum())
print("Weapon = Other or type unknown:", (df["Weapon"] == "Other or type unknown").sum())
print("Weapon = Weapon Not Reported:", (df["Weapon"] == "Weapon Not Reported").sum())

**What this shows:** Most columns have few hidden unknowns: `VicAge` 999 (1.5%), `VicSex` (0.2%), `VicRace` (1.3%) and unknown `Weapon` (about 6%). But `VicEthnic` is unknown in 55% of cases, because many agencies do not report it. So I will not use `VicEthnic`. For the other columns I will decide in Data Preparation if I remove these rows or keep "Unknown" as its own group.

### Data leakage check
I check if offender information is known only in solved cases. If yes, offender columns would give away the answer and cannot be used as features.

In [ ]:
unsolved_cases = df.loc[df["Solved"] == "No"]
solved_cases = df.loc[df["Solved"] == "Yes"]

print("Unsolved cases:", len(unsolved_cases))
print("Unsolved with OffAge = 999:", (unsolved_cases["OffAge"] == 999).sum())
print("Solved cases:", len(solved_cases))
print("Solved with OffAge = 999:", (solved_cases["OffAge"] == 999).sum())

**What this shows:** In unsolved cases, 99% have `OffAge` = 999 (unknown offender). In solved cases, only 5% have it. So offender information is almost only known when a case is solved. This confirms data leakage: offender columns (`OffAge`, `OffSex`, `OffRace`, `OffEthnic`, `OffCount`), and also `Relationship` and `Situation`, which depend on knowing the offender, must be removed before modelling.

### Duplicate IDs
The data dictionary says that victims of the same incident have the same ID. I check how many rows share an ID.

In [ ]:
print("Rows:", len(df))
print("Unique IDs:", df["ID"].nunique())
print("Rows with a repeated ID:", len(df) - df["ID"].nunique())

**What this shows:** There are 929,433 rows but only 884,633 different IDs, so 44,800 rows (about 5%) share an ID. These are not mistakes: the data dictionary says victims of the same incident have the same ID. I keep them, because each row is a different victim. But victims of the same incident have the same result, so this is a small risk when I split the data into train and test. I note it as a limitation.

### Number of cases per year
I check if the number of cases changes a lot between years, especially after 2021, where the share of unsolved cases dropped.

In [ ]:
plt.figure(figsize=(14, 4))
sns.countplot(data=df, x="Year", color="b")
plt.xticks(rotation=90)
plt.title("Number of cases per year")
plt.show()

**What this shows:** Most years have between 15,000 and 24,000 cases. The number goes up in 2020–2021 and then goes down to about 16,000 in 2025, which is similar to 2015–2019. So there is no sudden loss of data after 2021, and my idea that missing reports caused the drop in unsolved cases is not supported. The reason for the drop stays unclear. I note this as a limitation: the most recent years may behave differently from earlier years.

### "Weapon Not Reported"
This value is not in the data dictionary. I check in which years and from which source it appears.

In [ ]:
weapon_not_reported = df.loc[df["Weapon"] == "Weapon Not Reported"]

print(weapon_not_reported["Year"].value_counts().sort_index())
print(weapon_not_reported["Source"].value_counts())

**What this shows:** "Weapon Not Reported" appears only from 2020 to 2025 (2,187 cases), from both FBI and MAP sources. Before 2020 it does not exist. So it is probably a new code from a change in how police report data. It means the same as "Other or type unknown", so in Data Preparation I will merge them into one group.

### Summary: issues to fix in Data Preparation
The problems I found, from most to least important, and my plan for each.

| # | Issue | Plan |
|---|---|---|
| 1 | Data leakage: offender info is known only for solved cases (`OffAge` 999 in 99% of unsolved vs 5% of solved) | Remove offender columns, `Relationship`, `Situation`, `Circumstance`, `Subcircum` |
| 2 | 23,785 negligence cases (2.6%) | Remove, out of scope |
| 3 | ID and technical columns | Remove `ID`, `Ori`, `Agency`, `Incident`, `ActionType`, `FileDate`, `filter_$` |
| 4 | `VicEthnic`: 55% unknown, sensitive | Do not use |
| 5 | `VicRace` is sensitive | Not a feature, keep for fairness check |
| 6 | Hidden unknowns: `VicAge` 999 (1.5%), `Weapon` unknown (6%) | Decide in Data Preparation |
| 7 | "Weapon Not Reported" (only from 2020) | Merge with "Other or type unknown" |
| 8 | Very small weapon groups | Group together |
| 9 | Class imbalance (71% / 29%) | Stratified split, recall and F1 |
| 10 | 5% of rows share an ID | Keep, note as limitation |
| 11 | Unsolved share drops after 2021 | Note as limitation |

# 3. Data Preparation
In this section I fix the problems from the Data Quality Report. I keep the raw data in `df` and save all changes in a new table, `df_prep`. Every change is written in the preparation log at the end of this section.

### Step 1: Remove negligence cases
My question is about intentional homicides, so I remove cases of manslaughter by negligence.

In [22]:
n_rows_raw = len(df)

df_prep = df.copy().loc[df["Homicide"] == "Murder and non-negligent manslaughter"]

print("Rows before:", n_rows_raw)
print("Rows after:", len(df_prep))
print("Rows removed:", n_rows_raw - len(df_prep))
print("Fraction removed:", 1.0 - len(df_prep) / n_rows_raw)

Rows before: 929433
Rows after: 905648
Rows removed: 23785
Fraction removed: 0.025590870993390635


### Preparation log

| Issue | Action | Rationale / evidence | Assumption | Number affected | Risk / verification |
|---|---|---|---|---|---|
| Negligence cases out of scope | Removed rows where `Homicide` = "Manslaughter by negligence" | My question is about intentional killings; these are accidents | Negligence cases behave differently from murders | 23,785 rows (2.6%) | Small loss of data; checked row count before and after |